# Policy Search Example

This notebook demonstrates a simple Policy Search method using the REINFORCE algorithm.
The policy is represented explicitly and learned directly, without using a value function.

## Environment Description

We use a simple 1D environment with 3 states: 0, 1, 2.
State 2 is a terminal state.

Actions:
0: move left
1: move right

Reward:
Reaching state 2 gives reward +1.
All other transitions give reward 0.

In [1]:
# Số trạng thái trong môi trường
num_states = 3

# Số hành động
num_actions = 2

# Trạng thái kết thúc
terminal_state = 2

# Hàm mô phỏng một bước trong môi trường
def step(state, action):
    # Nếu đã ở trạng thái kết thúc thì không thay đổi trạng thái
    if state == terminal_state:
        return state, 0
    
    # Nếu hành động là sang trái
    if action == 0:
        next_state = max(0, state - 1)
    # Nếu hành động là sang phải
    else:
        next_state = min(num_states - 1, state + 1)
    
    # Phần thưởng chỉ xuất hiện khi tới trạng thái kết thúc
    reward = 1 if next_state == terminal_state else 0
    
    return next_state, reward

## Policy Representation

The policy is parameterized by a table of preferences ($\theta$ - bảng độ ưu tiên hành động).

What is a preference?

- A preference is not a probability.

- It is a numerical value that represents how much the policy “favors” an action in a given state.

- The larger the preference, the more likely that action is to be selected.

We use a softmax function to convert preferences into action probabilities:

$$ \pi(a \mid s) = \frac{\exp(\theta[s][a])}{\sum_{a'} \exp(\theta[s][a'])} $$


In [2]:
import math
import random

# Khởi tạo tham số policy (preferences) với giá trị 0
theta = [[0.0 for _ in range(num_actions)] for _ in range(num_states)]

# Hàm softmax để tính xác suất hành động
def softmax(preferences):
    # Tính e^x cho từng phần tử
    exp_values = [math.exp(p) for p in preferences]
    # Tính tổng các giá trị e^x
    total = sum(exp_values)
    # Chuẩn hóa để được phân phối xác suất
    return [v / total for v in exp_values]

# Hàm lấy xác suất hành động theo policy
def policy(state):
    # Áp dụng softmax lên tham số của trạng thái hiện tại
    return softmax(theta[state])

## Sampling Actions from the Policy

Actions are sampled according to the probability distribution given by the policy.

In [3]:
# Hàm chọn hành động theo policy
def choose_action(state):
    # Lấy phân phối xác suất của các hành động
    probs = policy(state)
    # Lấy một số ngẫu nhiên trong khoảng [0, 1)
    r = random.random()
    # Duyệt qua các hành động để chọn theo xác suất
    cumulative = 0.0                 # Khởi tạo tổng xác suất tích lũy
    for action, prob in enumerate(probs):  # Duyệt từng hành động và xác suất tương ứng
        cumulative += prob           # Cộng dồn xác suất
        if r < cumulative:           # Nếu số ngẫu nhiên r rơi vào khoảng này
            return action            # Chọn hành động hiện tại
    return num_actions - 1           # Trường hợp biên do sai số số học, chọn hành động cuối

## Policy Learning with REINFORCE

We update the policy parameters using the REINFORCE rule:

$$\theta = \theta + \alpha * G * \nabla(log(\pi(a|s)))$$

Where:

* $\theta$ denotes the parameters of the policy.
* $\alpha$ is the learning rate that controls the step size of the update.
* $G$ is the total reward obtained from the selected action (or from that time step onward).
* $\pi(a \mid s)$ is the probability of selecting action $a$ in state $s$ under the current policy.
* $\nabla \log(\pi(a \mid s))$ is the gradient (vector đạo hàm riêng) of the log-probability of the selected action with respect to the policy parameters $\theta$:
$$ \nabla \log(\pi(a \mid s)) \equiv \nabla_{\theta} \log(\pi(a \mid s; \theta)) $$

In [4]:
# Tốc độ học
alpha = 0.1

# Số episode huấn luyện
episodes = 30

# Vòng lặp huấn luyện policy
for episode in range(episodes):
    state = 0
    trajectory = []
    rewards = []
    
    # Sinh một episode theo policy hiện tại
    while state != terminal_state:
        action = choose_action(state)
        next_state, reward = step(state, action)
        trajectory.append((state, action))
        rewards.append(reward)
        state = next_state
    
    # Tổng phần thưởng của episode
    G = sum(rewards)
    
    # Cập nhật policy parameters
    for (state, action) in trajectory:
        probs = policy(state)
        for a in range(num_actions):
            if a == action:
                # Tăng xác suất của hành động đã chọn
                theta[state][a] += alpha * G * (1 - probs[a])
            else:
                # Giảm xác suất của các hành động khác
                theta[state][a] -= alpha * G * probs[a]

## Applying the Learned Policy

After training, we apply the learned policy greedily by selecting the action with highest probability.

In [5]:
# Áp dụng policy đã học
state = 0
terminal_state = terminal_state  # giả sử đã được định nghĩa trước
max_steps = 100                  # số bước tối đa để tránh lặp vô hạn

trajectory = [state]  # Lưu lại đường đi của agent
actions = []          # Lưu chuỗi các hành động mà agent thực hiện

step_count = 0

while state != terminal_state and step_count < max_steps:
    # Lấy xác suất hành động từ policy
    probs = policy(state)
    
    # Chọn hành động có xác suất cao nhất (greedy)
    action = probs.index(max(probs))
    
    # Thực hiện hành động và chuyển sang trạng thái mới
    state, _ = step(state, action)
    
    trajectory.append(state)
    actions.append(action)
    
    # Tăng bộ đếm số bước
    step_count += 1

print("Agent trajectory:", trajectory)   # In đường đi của agent
print("Sequence of actions:", actions)   # In chuỗi các hành động của agent

# Thông báo lý do kết thúc
if state == terminal_state:
    print("Episode ended: reached terminal state.")
else:
    print("Episode ended: reached maximum number of steps.")

Agent trajectory: [0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0]
Sequence of actions: [1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0]
Episode ended: reached maximum number of steps.


## Conclusion

This example illustrates Policy Search, where the policy is learned directly.
Unlike value-based methods, Policy Search does not rely on a Q-function and instead optimizes policy parameters directly.